In [ ]:
import json
from pathlib import Path
import pandas as pd


### Utilities to dynamically generate categories for tags

In [16]:
categories_path = Path("../data/recipes/processed/recipe_categories.json")

## Read category Info from a predefined JSON file
def read_category_info(category_info_path: Path) -> pd.DataFrame:
    with open(category_info_path, "r") as f:
        category_info = json.load(f)
    return category_info

category_info = read_category_info(categories_path)

In [19]:
category_info

[{'category': 'Course',
  'tags': [{'tag_id': 22, 'tag_name': 'breakfast'},
   {'tag_id': 47, 'tag_name': 'brunch'},
   {'tag_id': 23, 'tag_name': 'main-dish'},
   {'tag_id': 9, 'tag_name': 'side-dishes'},
   {'tag_id': 70, 'tag_name': 'appetizers'},
   {'tag_id': 85, 'tag_name': 'desserts'},
   {'tag_id': 86, 'tag_name': 'lunch'},
   {'tag_id': 87, 'tag_name': 'snacks'},
   {'tag_id': 159, 'tag_name': 'salads'},
   {'tag_id': 116, 'tag_name': 'soups-stews'},
   {'tag_id': 131, 'tag_name': 'sauces'},
   {'tag_id': 180, 'tag_name': 'dips'},
   {'tag_id': 207, 'tag_name': 'spreads'}]},
 {'category': 'Cuisine & Region',
  'tags': [{'tag_id': 25, 'tag_name': 'american'},
   {'tag_id': 8, 'tag_name': 'north-american'},
   {'tag_id': 11, 'tag_name': 'mexican'},
   {'tag_id': 134, 'tag_name': 'asian'},
   {'tag_id': 135, 'tag_name': 'indian'},
   {'tag_id': 219, 'tag_name': 'chinese'},
   {'tag_id': 313, 'tag_name': 'japanese'},
   {'tag_id': 339, 'tag_name': 'thai'},
   {'tag_id': 172, 'tag_

In [17]:
## Given a list of recipe IDs, returns an SQL query string to fetch associated tags
def get_tag_lookup_query(recipe_ids: list[int]) -> str:
    recipe_ids_str = ",".join([str(rid) for rid in recipe_ids])
    query = "SELECT tags.tag_id, tags.tag_name, count(tags.tag_id) FROM recipe_tags, tags WHERE recipe_tags.recipe_id in ({recipe_ids_str}) and tags.tag_id=recipe_tags.tag_id group by tags.tag_id;" .format(recipe_ids_str=recipe_ids_str)
    return query


In [18]:
# testing
recipe_ids = [413, 53680, 49865, 47264, 573727, 482245, 583, 4772, 23475]
get_tag_lookup_query(recipe_ids)

'SELECT tags.tag_id, tags.tag_name, count(tags.tag_id) FROM recipe_tags, tags WHERE recipe_tags.recipe_id in (413,53680,49865,47264,573727,482245,583,4772,23475) and tags.tag_id=recipe_tags.tag_id group by tags.tag_id;'

In [ ]:
# Given a dataframe of tags with their counts, return categories with tags.
def get_tags_for_qa(tags_from_recipes: pd.DataFrame, category_info: dict) -> list[dict]:
    qa_tags = []
    for category_item in category_info:
        category_tags = []
        category = category_item["category"]
        tags = category_item["tags"]
        for tag in tags:
            tag_id = tag['tag_id']
            if tag_id in tags_from_recipes["tag_id"].values:
                category_tags.append(tag)
        if len(category_tags) > 0:
            qa_tags.append({"category": category, "tags": category_tags})
    return qa_tags

In [25]:
# testing
tags_from_recipes = pd.read_csv("../data/recipes/test/tags_for_recipes.csv")
get_tags_for_qa(tags_from_recipes, category_info)

[{'category': 'Course',
  'tags': [{'tag_id': 22, 'tag_name': 'breakfast'},
   {'tag_id': 23, 'tag_name': 'main-dish'},
   {'tag_id': 9, 'tag_name': 'side-dishes'},
   {'tag_id': 85, 'tag_name': 'desserts'}]},
 {'category': 'Cuisine & Region',
  'tags': [{'tag_id': 25, 'tag_name': 'american'},
   {'tag_id': 8, 'tag_name': 'north-american'}]},
 {'category': 'Diet & Health',
  'tags': [{'tag_id': 98, 'tag_name': 'healthy'},
   {'tag_id': 127, 'tag_name': 'low-saturated-fat'}]},
 {'category': 'Season & Weather',
  'tags': [{'tag_id': 13, 'tag_name': 'fall'},
   {'tag_id': 19, 'tag_name': 'seasonal'},
   {'tag_id': 155, 'tag_name': 'summer'}]},
 {'category': 'Occasion & Holiday',
  'tags': [{'tag_id': 7, 'tag_name': 'occasion'},
   {'tag_id': 14, 'tag_name': 'holiday-event'}]},
 {'category': 'Preparation Method',
  'tags': [{'tag_id': 26, 'tag_name': 'oven'},
   {'tag_id': 110, 'tag_name': 'deep-fry'},
   {'tag_id': 110, 'tag_name': 'deep-fry'}]},
 {'category': 'Lifestyle & Convenience',
 

### Filter Queries: 
Multi-condition filter queries for follow-up 
Filters include: 
1) one or more tags (from categories): IDs
2) cook time
3) nutrition info: calories, saturated_fat, total_fat, protein, sugar
4) ingredient count
5) number of steps
6) average rating

In [34]:
def build_recipe_filter_query(
    recipe_ids=None,
    tag_ids=None,                      # list[int]
    cook_time_max=None,                # int
    ingredient_count_max=None,         # int
    num_steps_max=None,                # int
    avg_rating_min=None,               # float
    calories_max=None,                 # float
    saturated_fat_max=None,            # float
    total_fat_max=None,                # float
    protein_min=None,                  # float
    sugar_max=None                     # float
):
    """
    Build SQL query dynamically based on optional filters.
    Only the required JOINs are included.
    """

    # Base SELECT
    select_clause = """
SELECT DISTINCT r.recipe_id
FROM recipes r
"""

    joins = []
    where = []
    ctes = []

    # -------------------------
    # TAG FILTERS (via CTE)
    # -------------------------
    if tag_ids:
        tag_id_list = ", ".join(str(t) for t in tag_ids)

        ctes.append(f"""
recipe_filtered_by_tags AS (
    SELECT rt.recipe_id
    FROM recipe_tags rt
    WHERE rt.tag_id IN ({tag_id_list})
    GROUP BY rt.recipe_id
    HAVING COUNT(*) = {len(tag_ids)}
)
""")

        # Join the CTE
        joins.append("JOIN recipe_filtered_by_tags rf ON rf.recipe_id = r.recipe_id")

    # -------------------------------------
    # OPTIONAL: recipe_ids filtering
    # -------------------------------------
    if recipe_ids:
        ids_str = ", ".join(str(i) for i in recipe_ids)
        where.append(f"r.recipe_id IN ({ids_str})")

    # -------------------------
    # COOK TIME
    # -------------------------
    if cook_time_max is not None:
        where.append(f"r.cook_time_min <= {cook_time_max}")

    # -------------------------
    # INGREDIENT COUNT
    # -------------------------
    if ingredient_count_max is not None:
        where.append(f"r.ingredient_count <= {ingredient_count_max}")

    # -------------------------
    # STEP COUNT
    # -------------------------
    if num_steps_max is not None:
        where.append(f"r.num_steps <= {num_steps_max}")

    # -------------------------
    # AVERAGE RATING
    # -------------------------
    if avg_rating_min is not None:
        where.append(f"r.avg_rating >= {avg_rating_min}")

    # -------------------------
    # NUTRITION (conditional JOIN)
    # -------------------------
    nutrition_filters = []

    if calories_max is not None:
        nutrition_filters.append(f"n.calories <= {calories_max}")

    if saturated_fat_max is not None:
        nutrition_filters.append(f"n.saturated_fat_pdv <= {saturated_fat_max}")

    if total_fat_max is not None:
        nutrition_filters.append(f"n.total_fat_pdv <= {total_fat_max}")

    if protein_min is not None:
        nutrition_filters.append(f"n.protein_pdv >= {protein_min}")

    if sugar_max is not None:
        nutrition_filters.append(f"n.sugar_pdv <= {sugar_max}")

    # add join only if nutrition filter requested
    if nutrition_filters:
        joins.append("JOIN nutrition n ON n.recipe_id = r.recipe_id")
        where.extend(nutrition_filters)

    # Final assembly
    cte_sql = ""
    if ctes:
        cte_sql = "WITH " + ",".join(ctes)

    join_sql = "\n".join(joins)
    where_sql = ""
    if where:
        where_sql = "WHERE " + " AND ".join(where)

    sql = f"""
{cte_sql}{select_clause}{join_sql}
{where_sql};
"""

    # Strip leading whitespace
    s = "\n".join(line.rstrip() for line in sql.split("\n"))
    s = s.replace("\n\n", "\n")  
    return s

In [38]:
query = build_recipe_filter_query(
    recipe_ids=[38955,403593,420839,388546],
    tag_ids=[51, 162, 22, 127],
    cook_time_max=90,
    calories_max=120,
    ingredient_count_max=20,
)

print(query)


WITH
recipe_filtered_by_tags AS (
    SELECT rt.recipe_id
    FROM recipe_tags rt
    WHERE rt.tag_id IN (51, 162, 22, 127)
    GROUP BY rt.recipe_id
    HAVING COUNT(*) = 4
)
SELECT DISTINCT r.recipe_id
FROM recipes r
JOIN recipe_filtered_by_tags rf ON rf.recipe_id = r.recipe_id
JOIN nutrition n ON n.recipe_id = r.recipe_id
WHERE r.recipe_id IN (38955, 403593, 420839, 388546) AND r.cook_time_min <= 90 AND r.ingredient_count <= 20 AND n.calories <= 120;



In [39]:
query = build_recipe_filter_query(
    recipe_ids=[38955,403593,420839,388546],
    tag_ids=[51, 162, 22, 127],
    cook_time_max=90,
    ingredient_count_max=20,
)

print(query)


WITH
recipe_filtered_by_tags AS (
    SELECT rt.recipe_id
    FROM recipe_tags rt
    WHERE rt.tag_id IN (51, 162, 22, 127)
    GROUP BY rt.recipe_id
    HAVING COUNT(*) = 4
)
SELECT DISTINCT r.recipe_id
FROM recipes r
JOIN recipe_filtered_by_tags rf ON rf.recipe_id = r.recipe_id
WHERE r.recipe_id IN (38955, 403593, 420839, 388546) AND r.cook_time_min <= 90 AND r.ingredient_count <= 20;



In [40]:
query = build_recipe_filter_query(
    recipe_ids=[38955,403593,420839,388546],
    cook_time_max=90,
    ingredient_count_max=20,
)

print(query)


SELECT DISTINCT r.recipe_id
FROM recipes r
WHERE r.recipe_id IN (38955, 403593, 420839, 388546) AND r.cook_time_min <= 90 AND r.ingredient_count <= 20;



In [ ]:
query = build_recipe_filter_query(
    recipe_ids=[38955,403593,420839,388546],
    cook_time_max=90,
    ingredient_count_max=20,
    calories_max=120
)

print(query)


SELECT DISTINCT r.recipe_id
FROM recipes r
JOIN nutrition n ON n.recipe_id = r.recipe_id
WHERE r.recipe_id IN (38955, 403593, 420839, 388546) AND r.cook_time_min <= 90 AND r.ingredient_count <= 20 AND n.calories <= 150;

